In [2]:
import numpy as np 
import pandas as pd
from scipy.stats import norm

In [3]:
R = 2000
seed = 1234
rng = np.random.default_rng(seed)

#function to find pvalue, zvalue, and se

def report(theta_hat, boot, theta0, scale=1.0, unit=""):
    se = np.std(boot, ddof=1)
    z = (theta_hat - theta0) / se
    p = 2 * norm.cdf(-abs(z))
    print(f"seed = {seed}, R = {R}")
    print(f"estimate = {theta_hat:.6f}   SE = {se:.6f}")
    print(f"z = {z:.4f}   p = {p:.3g}")
    for conf, zc, alpha in [(90, 1.645, .10), (95, 1.96, .05), (99, 2.576, .01)]:
        lo, hi = theta_hat - zc * se, theta_hat + zc * se
        reject_test = p < alpha
        reject_ci = not (lo <= theta0 <= hi)
        print(f"{conf}% CI: ({lo:.6f}, {hi:.6f})"
              + (f"  = ({lo*scale:.3f}, {hi*scale:.3f}) {unit}" if scale != 1.0 else "")
              + f" | reject at alpha={alpha}: {reject_test} | CI excludes theta0: {reject_ci}"
              + f" | agree: {reject_test == reject_ci}")
    return se, z, p

In [4]:
#4

#generating the bootstrap estimates

df = pd.read_csv("./data/ames_prices.csv")
x = (df["Central.Air"] == "Y").astype(int).to_numpy()
n = len(x)

#estimate from the real set of data

theta_hat = x.mean()

boot = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(R)])
report(theta_hat, boot, theta0=0.90)

seed = 1234, R = 2000
estimate = 0.933106   SE = 0.004747
z = 6.9739   p = 3.08e-12
90% CI: (0.925297, 0.940915) | reject at alpha=0.1: True | CI excludes theta0: True | agree: True
95% CI: (0.923801, 0.942410) | reject at alpha=0.05: True | CI excludes theta0: True | agree: True
99% CI: (0.920877, 0.945334) | reject at alpha=0.01: True | CI excludes theta0: True | agree: True


(np.float64(0.004747105716837866),
 np.float64(6.973891887504447),
 np.float64(3.082910328310913e-12))

We can reject H0: p= 0.90 at all of the confidence intervals with 93.3% of Ames homes having central air. The 95% confidence interval is (0.923801, 0.942410) and 0.90 does not fall within it

In [5]:
#5
#correlation between area and price
df = pd.read_csv("./data/ames_prices.csv")
d = df[["area", "price"]].dropna()
x = np.arcsinh(d["area"].to_numpy())
y = np.arcsinh(d["price"].to_numpy())
n = len(x)



theta_hat = np.corrcoef(x, y)[0, 1]

boot = np.empty(R)


for r in range(R):
    idx = rng.integers(0, n, size=n)   
    boot[r] = np.corrcoef(x[idx], y[idx])[0, 1]

report(theta_hat, boot, theta0=0.0)


seed = 1234, R = 2000
estimate = 0.723166   SE = 0.010215
z = 70.7961   p = 0
90% CI: (0.706363, 0.739969) | reject at alpha=0.1: True | CI excludes theta0: True | agree: True
95% CI: (0.703145, 0.743187) | reject at alpha=0.05: True | CI excludes theta0: True | agree: True
99% CI: (0.696853, 0.749479) | reject at alpha=0.01: True | CI excludes theta0: True | agree: True


(np.float64(0.01021477530027918),
 np.float64(70.79609014888177),
 np.float64(0.0))

We can reject H0: p = 0 at all levels as 0 does not fall within any of the intervals

In [6]:
#6
#
df = pd.read_csv("./data/metabric.csv")
x = df["Overall Survival (Months)"].dropna().to_numpy()
n = len(x)

theta_hat = x.mean()
theta0 = 120

boot = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(R)])
report(theta_hat, boot, theta0=120, scale=1/12, unit="years")

seed = 1234, R = 2000
estimate = 129.094291   SE = 2.169510
z = 4.1919   p = 2.77e-05
90% CI: (125.525448, 132.663135)  = (10.460, 11.055) years | reject at alpha=0.1: True | CI excludes theta0: True | agree: True
95% CI: (124.842052, 133.346531)  = (10.404, 11.112) years | reject at alpha=0.05: True | CI excludes theta0: True | agree: True
99% CI: (123.505634, 134.682949)  = (10.292, 11.224) years | reject at alpha=0.01: True | CI excludes theta0: True | agree: True


(np.float64(2.169509878894695),
 np.float64(4.191864474050856),
 np.float64(2.7667126379331673e-05))

the estimate is 129.1 months or 129.1/12 = 10.76 years. We can reject H0: p = 120, it does not fall within any of the confidence intervals

In [7]:
df = pd.read_csv("./data/nhanes_data_17_18.csv")
d = df[(df["AgeInYearsAtScreening"] >= 18) & df["SleepHoursWeekdaysOrWorkdays"].notna()]
x = d["SleepHoursWeekdaysOrWorkdays"].to_numpy()
n = len(x)
print("n =", n)

theta_hat = x.mean()
boot = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(R)])
report(theta_hat, boot, theta0=8)

n = 5489
seed = 1234, R = 2000
estimate = 7.621242   SE = 0.022808
z = -16.6062   p = 6.28e-62
90% CI: (7.583723, 7.658762) | reject at alpha=0.1: True | CI excludes theta0: True | agree: True
95% CI: (7.576538, 7.665946) | reject at alpha=0.05: True | CI excludes theta0: True | agree: True
99% CI: (7.562489, 7.679996) | reject at alpha=0.01: True | CI excludes theta0: True | agree: True


/var/folders/sd/qls438lx2_q9wdqdv6nztvrm0000gn/T/ipykernel_38447/3949875755.py:1: DtypeWarning: Columns (142) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./data/nhanes_data_17_18.csv")


(np.float64(0.022808168859565443),
 np.float64(-16.60622197959631),
 np.float64(6.282756379483108e-62))

statistical significance sees the gap as noise. Practical significance asks wheter the gap matters or not. The mean sleep is 7.6 hours. Its 95 CI is (7.57, 7.66) at theta0 = 8. We can reject th hypothesis at this question as well